| Roll No.| Subject Code | Assignment No. |
|:-------:|:------------:|:--------------:|
| 2650026 | CS69101      | Lab 3          |

To construct an AVL Tree for a given set of elements stored in a file, implement insertion and deletion operations on the constructed tree, and write the contents of the tree (after these operations) into a new file using in-order traversal.

Prerequisite Code Section
----------------------------

In [ ]:
import sys
from io import TextIOWrapper
from typing import Self
import random

prep_arr = [random.randint(0, i*10) for i in range(10000)]
prep_arr.sort(reverse=True) # Reverse Sorted Array for Worst Case Scenario
with open("input0.txt", "w") as f:
    for i in prep_arr:
        f.write(f"{i}, ")
    f.write("\n")

# Q1

Read a set of integer elements from an input file `(input.txt)`.

In [ ]:
# Read a set of integer elements from an input file (input.txt)
arr0 = []
data0 = open("input.txt", 'r')

try:
    for line in data0.readlines():
        line_data = line.rstrip().rstrip(',').replace(' ','').split(',') #using rstrip to remove the \n
        for data in line_data:
            arr0.append(int(data))
    data0.close()
except Exception as e:
    sys.exit("[ERROR] Invalid data" + str(e))

In [ ]:
print(arr0)

# Q2 & Q3

Construct a balanced AVL tree by inserting these elements one by one, performing the necessary LL, RR, LR, RL rotations to maintain balance after every insertion.

Implement an insert(key) operation that adds a new node to the AVL tree while preserving
the AVL balance property $$ ( \, \text{balance factor} \in \{ -1, 0, 1 \} \text{ for every node} ) \,. $$

In [ ]:
class AVL_Node: # AVL Node class
    def __init__(self):
        self.__lchild = None
        self.__rchild = None
        self.__value = None
        self.__parent = None
        self.__height = None
        self.__balance_factor = None

    def _get_height(self) -> int | None:
        if self.__height is None:
            self.__height = 1 + max(
                (self.__lchild._get_height() if self.__lchild else -1), # type: ignore
                # Height of empty tree set to be -1
                (self.__rchild._get_height() if self.__rchild else -1) # type: ignore
            )
        return self.__height

    def _set_height(self, height: int) -> None:
        self.__height = height

    height = property(_get_height, _set_height, doc="Height of the node")

    def _get_balance_factor(self) -> int | None:
        if self.__balance_factor is None:
            self.__balance_factor = (
                (self.__lchild._get_height() if self.__lchild else -1) -
                (self.__rchild._get_height() if self.__rchild else -1)
            ) # type: ignore
        return self.__balance_factor

    balance_factor = property(_get_balance_factor, doc="Balance factor of the node")

    def _get_lchild(self) -> Self | None:
        return self.__lchild

    def _set_lchild(self, lchild: Self | None) -> None: # type: ignore
        self.__lchild = lchild
        if lchild is not None:
            lchild.__parent = self

    lchild = property(_get_lchild, _set_lchild, doc="Left child of the node")

    def _get_rchild(self) -> Self | None:
        return self.__rchild

    def _set_rchild(self, rchild: Self | None) -> None: # type: ignore
        self.__rchild = rchild
        if rchild is not None:
            rchild.__parent = self

    rchild = property(_get_rchild, _set_rchild, doc="Right child of the node")

    def _get_parent(self) -> Self | None:
        return self.__parent

    def _set_parent(self, parent: Self | None) -> None: # type: ignore
        self.__parent = parent

    parent = property(_get_parent, _set_parent, doc="Parent of the node")

    def _get_value(self) -> int | None:
        return self.__value

    def _set_value(self, value: int) -> None:
        self.__value = value

    value = property(_get_value, _set_value, doc="Value of the node")

    def __str__(self) -> str:
        return f"Node(value={self.__value}, height={self.__height}, balance_factor={self.__balance_factor})"

    def __repr__(self) -> str:
        return self.__str__()

In [ ]:
def AVL_right_rotate(k1_node: AVL_Node | None, k2_node: AVL_Node | None) -> AVL_Node | None:
    if k1_node is None or k2_node is None:
        return

    # print(f"Right Rotate: k1_node={k1_node}, k2_node={k2_node}")

    # From Weiss' Data Structures and Algorithm Analysis in C++ (Page 156)
    TEMP_NODE = k2_node.rchild
    k1_node.rchild = k2_node
    k2_node.lchild = TEMP_NODE

    # Update Parents
    if k1_node.rchild is not None:
        k1_node.rchild.parent = k1_node
    if k2_node.lchild is not None:
        k2_node.lchild.parent = k2_node

    k1_node._set_height(k1_node._get_height() if k1_node._get_height() is not None else 0) # type: ignore
    k2_node._set_height(k2_node._get_height() if k2_node._get_height() is not None else 0) # type: ignore

    return k1_node

def AVL_left_rotate(k1_node: AVL_Node | None, k2_node: AVL_Node | None) -> AVL_Node | None:
    if k1_node is None or k2_node is None:
        return

    # print(f"Left Rotate: k1_node={k1_node}, k2_node={k2_node}")

    # From Weiss' Data Structures and Algorithm Analysis in C++ (Page 156)
    TEMP_NODE = k2_node.lchild
    k1_node.lchild = k2_node
    k2_node.rchild = TEMP_NODE

    # Update Parents
    if k1_node.lchild is not None:
        k1_node.lchild.parent = k1_node
    if k2_node.rchild is not None:
        k2_node.rchild.parent = k2_node
    
    k1_node._set_height(k1_node._get_height() if k1_node._get_height() is not None else 0) # type: ignore
    k2_node._set_height(k2_node._get_height() if k2_node._get_height() is not None else 0) # type: ignore
    return k1_node

def AVL_double_rotate_right(k1_node: AVL_Node | None, k3_node: AVL_Node | None) -> AVL_Node | None:
    if k1_node is None or k3_node is None:
        return

    # print(f"Double Right Rotate: k1_node={k1_node}, k3_node={k3_node}")

    k2_node = k1_node.rchild
    k1_node = AVL_left_rotate(k2_node, k1_node)
    k3_node = AVL_right_rotate(k1_node, k3_node)
    return k3_node

def AVL_double_rotate_left(k1_node: AVL_Node | None, k3_node: AVL_Node | None) -> AVL_Node | None:
    if k1_node is None or k3_node is None:
            return

    # print(f"Double Left Rotate: k1_node={k1_node}, k3_node={k3_node}")

    k2_node = k1_node.lchild
    k1_node = AVL_right_rotate(k2_node, k1_node)
    k3_node = AVL_left_rotate(k1_node, k3_node)
    return k3_node

def AVL_insert(root: AVL_Node | None, value: int) -> AVL_Node | None:
    if (root is None) or (root.value is None):                              # NULL type handling && Base Case
        new_node = AVL_Node()
        new_node.value = value
        new_node.height = 0
        print(f"Created {new_node}") 
        return new_node

    if value < root.value:                                                  # X < MID
        root.lchild = AVL_insert(root.lchild, value)
        print(f"Inserted {root.lchild} as left child of {root}")
        if root.balance_factor == 2:
            if value < root.lchild.value: # type: ignore
                root = AVL_right_rotate(root.lchild, root)                  #LL
            else:
                root = AVL_double_rotate_right(root.lchild, root)           #LR
    elif value > root.value:                                                # X > MID
        root.rchild = AVL_insert(root.rchild, value)
        print(f"Inserted {root.rchild} as right child of {root}")
        if root.balance_factor == -2:
            if value > root.rchild.value: # type: ignore
                root = AVL_left_rotate(root.rchild, root)                   #RR
            else:
                root = AVL_double_rotate_left(root.rchild, root)            #RL
    return root

In [ ]:
root = AVL_Node()
for i in arr0:
    root = AVL_insert(root, i)

# Q4 & Q5

Perform the in-order traversal of the final AVL tree and write the traversal output to a new file `(output.txt)`.

Display the height and balance factor of the tree before and after each operation (optional enhancement).

In [ ]:
def AVL_inorder_traversal(root: AVL_Node | None, out_file: TextIOWrapper | None) -> None:
    if root is None:
        return
    AVL_inorder_traversal(root.lchild, out_file = out_file)
    if out_file is not None:
        print(root.value, end = ', ', file = out_file)
    print(root.value, end = ', ')
    print(root)
    AVL_inorder_traversal(root.rchild, out_file = out_file)

def AVL_preorder_traversal(root: AVL_Node | None, out_file: TextIOWrapper | None) -> None:
    if root is None:
        return
    if out_file is not None:
        print(root.value, end = ', ', file = out_file)
    print(root.value, end = ', ')
    print(root)
    AVL_preorder_traversal(root.lchild, out_file)
    AVL_preorder_traversal(root.rchild, out_file)

def AVL_postorder_traversal(root: AVL_Node | None, out_file: TextIOWrapper | None) -> None:
    if root is None:
        return
    AVL_postorder_traversal(root.lchild, out_file)
    AVL_postorder_traversal(root.rchild, out_file)
    if out_file is not None:
        print(root.value, end = ', ', file = out_file)
    print(root.value, end=', ')
    print(root)

In [ ]:
data1 = open("output.txt", 'w')
AVL_inorder_traversal(root, data1)
data1.close()

# Q6

Plot Height vs number of node

# Q7

Compare AVL and BST search time